# TODO: CHECK PREPROCESSING AGAIN

# Supervised Machine Learning - specific Preprocessing
This notebook contains code to preprocess the articles in a way specifically tailored to supervised machine learning.

Since SML models in this project use term frequency - inverse document frequency, we stem the words to their lexical root form and get rid of unwanted noise.

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [1]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()
print(f"Root directory found: {root_dir}")

# add the root directory to the system path
sys.path.append(root_dir)
if root_dir in sys.path:
    print(f"Root directory added to system path.")


# import custom modules
from src import data

Root directory found: c:\dev\automated_title_abstract_screening
Root directory added to system path.


## Import Datasets
We begin by importing the dataset dictionary as always:

In [2]:
from src import data

data_directory = '../../data/datasets/04_preprocessed'
datasets = data.dict_from_directory(data_directory, type='polars')

## Helper Functions
Here we define some helper functions to preprocess the datasets:

In [3]:
from bs4 import BeautifulSoup

def remove_html(text: str)-> str:
    """Remove html tags from a string
    
    Args:
    text: str: a string containing html tags

    Returns:
    str: a string without html tags
    """
    if not text or text.strip() == "":
        return ""
    
    try:
        return BeautifulSoup(text, 'html.parser').get_text()
    except Exception as e:
        print(f"Error removing HTML from text: {text[:50]}... - {e}")
        return text  # Return original text if HTML removal fails

In [4]:
import re

def remove_special_characters(text:str, digits_also:bool=True,) -> str:
    """Remove special characters from a string
    
    Args:
    text: str: a string containing special characters
    digits_also: bool: if True, digits are also removed, otherwise only special characters are removed

    Returns:
    str: a string without special characters
    """
    if not text or text.strip() == "":
        return ""
    
    try:
        expression = '[^ A-Za-z]+' if digits_also else '[^ A-Za-z0-9]+'
        specials_removed = re.sub(expression, ' ', text)
        return ' '.join(word.strip() for word in specials_removed.split())
    except Exception as e:
        print(f"Error removing special characters from text: {text[:50]}... - {e}")
        return text  # Return original text if special character removal fails

In [5]:
from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

In [6]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def lemmatize_text(input: str) -> str:
    """Lemmatize a text

    Args:
        input: str: a string to lemmatize

    Returns:
        str: a lemmatized string
    """
    # Handle empty or None input
    if not input or input.strip() == "":
        return ""
    
    try:
        lemmatizer = WordNetLemmatizer()
        tokens = word_tokenize(input)
        pos_tags = nltk.pos_tag(tokens)
        
        output = []
        for word, tag in pos_tags:
            wordnet_pos = get_wordnet_pos(tag)
            if wordnet_pos:
                lemmatized_word = lemmatizer.lemmatize(word, pos=wordnet_pos)
            else:
                lemmatized_word = lemmatizer.lemmatize(word)
            output.append(lemmatized_word)

        return ' '.join(output)
    except Exception as e:
        print(f"Error lemmatizing text: {input[:50]}... - {e}")
        return input  # Return original text if lemmatization fails


In [7]:
from num2words import num2words

def digits_to_words(match):
  """
  Convert string digits to the English words. The function distinguishes between
  cardinal and ordinal.
  E.g. "2" becomes "two", while "2nd" becomes "second"

  Input: str
  Output: str
  """
  suffixes = ['st', 'nd', 'rd', 'th']
  # Making sure it's lower cased so not to rely on previous possible actions:
  string = match[0].lower()
  if string[-2:] in suffixes:
    type='ordinal'
    string = string[:-2]
  else:
    type='cardinal'

  return num2words(string, to=type)

In [8]:
from nltk.corpus import stopwords

def remove_stop_words(text):
    """
    Remove stopwords.

    Input: str
    Output: str
    """
    if not text or text.strip() == "":
        return ""
    
    try:
        stopwords_set = set(stopwords.words('english'))
        return " ".join([word for word in text.split() if word not in stopwords_set])
    except Exception as e:
        print(f"Error removing stop words from text: {text[:50]}... - {e}")
        return text  # Return original text if stop word removal fails

In [9]:
import polars as pl

nltk.download('averaged_perceptron_tagger_eng') # part-of-speech tagging
nltk.download('wordnet') # lemmatization
nltk.download('punkt') # tokenization
nltk.download('stopwords') # stopwords

import warnings
warnings.filterwarnings("ignore")

def preprocess_dataframe(dataframe: pl.DataFrame) -> pl.DataFrame:
    """
    Preprocess a dataframe by filling null values, converting text to lowercase,
    and applying a series of text preprocessing steps including removing html tags, 
    special characters, spelling mistakes, lemmatizing (using part-of-speech-tags), and removing stopwords.

    Args:
    dataframe: pl.DataFrame: a dataframe with columns 'title' and 'abstract' containing text to preprocess

    Returns:
    pl.DataFrame: a dataframe with  preprocessed text
    """
    return dataframe.with_columns(
        pl.col("title")
        .fill_null(" ")  # Handle null values
        .str.to_lowercase()  # Vectorized operation
        .map_elements(remove_html, return_dtype=pl.String)  # Custom function for complex logic
        .map_elements(lemmatize_text, return_dtype=pl.String)
        .map_elements(remove_special_characters, return_dtype=pl.String)
        .map_elements(lambda x: re.sub(r'\d+(st)?(nd)?(rd)?(th)?', digits_to_words, x), return_dtype=pl.String)
        .map_elements(remove_stop_words, return_dtype=pl.String),
        pl.col('abstract')
        .fill_null(" ")  # Handle null values
        .str.to_lowercase()  # Vectorized operation
        .map_elements(remove_html, return_dtype=pl.String)  # Custom function for complex logic
        .map_elements(lemmatize_text, return_dtype=pl.String)
        .map_elements(remove_special_characters, return_dtype=pl.String)
        .map_elements(lambda x: re.sub(r'\d+(st)?(nd)?(rd)?(th)?', digits_to_words, x), return_dtype=pl.String)
        .map_elements(remove_stop_words, return_dtype=pl.String),
    )

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mfaig\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mfaig\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mfaig\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mfaig\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Preprocess and Export

In [10]:
from tqdm.notebook import tqdm

data_directory_preprocessing = '../../data/datasets/04_preprocessed/supervised'

print('Preprocessing datasets...')
for subject, dataset in tqdm(datasets.items(), total=len(datasets)):
    datasets[subject] = preprocess_dataframe(dataset)

    datasets[subject].write_csv(f'{data_directory_preprocessing}/{subject}_preprocessed.csv')
print('Done!')

Preprocessing datasets...


  0%|          | 0/5 [00:00<?, ?it/s]

Error lemmatizing text: the effectiveness of clonidine as an analgesic in ... - 'WordNetCorpusReader' object has no attribute '_LazyCorpusLoader__args'
Done!
